### Método $\theta$

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurimendiluce/AN2026/blob/main/diferencias_finitas/clase_3.ipynb)

Consideramos el siguiente problema:

$$\begin{align}
u_t&=u_{xx}\\
u(0,t)&=u(1,t)=0\\
u(x,0)&=u_0(x)
\end{align}$$

Se propone el siguiente método:
$$\frac{U^{n+1}_j-U^{n}_j}{\Delta t}=(1-\theta)\frac{U^{n}_{j+1}-2U^{n}_{j}+U^{n}_{j-1}}{(\Delta x)^2}+\theta\frac{U^{n+1}_{j+1}-2U^{n+1}_{j}+U^{n+1}_{j-1}}{(\Delta x)^2}$$

In [ ]:
using LinearAlgebra
using Plots

Tomar el intervalo temporal $[0,1]$ e implementar el método $\theta$. La función deberá tomar como parametros $u_0(x)$, $\theta$, $r$, $N$ (particiones del intervalo) y la cantidad de pasos.

In [ ]:
Uo(x) = x .* (1 .- x)


f (generic function with 2 methods)

In [ ]:
# Construir U inicial y realizar la iteración.
function metodo_theta(Uo,r,θ,N;steps = 100)
    Δx = 1/N
    Δt = r*Δx^2

    # Nodos interiores: x_j = j h, j=1,...,J, con x=0 y x=1 como frontera.
    J = round(Int, 1 / Δx) - 1
    x_grid = collect(1:J) .* Δx
    
    U = Uo(x_grid)
    
    A = Tridiagonal(ones(J-1), -2 .* ones(J), ones(J-1))
    B = I - θ * r * A
    C = I + (1 - θ) * r * A
    
    n=0
    while n<steps
        U = B \ (C * U)
        n += 1
    end

    p = plot(x_grid, U, xlabel="x", ylabel="U(x,t)", label="θ = $θ")
    display(p)
    println("El valor de r es: ", r)
    return U
end

U = metodo_theta(Uo,0.5,0.5,10);

Ahora realicemos distintas simulaciones, para testear la estabilidad o inestabilidad para el método explicito e implicito.

In [ ]:
# Ejemplos:
#metodo_theta(Uo,0.55,0.0, 20)   # Euler explícito
#metodo_theta(Uo,0.55,1, 20)   # Euler implícito


Tomar $\theta=0.5$ y verificar la estabilidad independientemente del $r$. Observar lo que sucede si perturbo sensiblemente dicho valor de modo que $\theta<0.5$.

In [ ]:
# Usar la función de arriba con distintos valores de θ y r.
# Por ejemplo, comparar θ = 0.5 con θ = 0.49.

### Estabilidad del método $\theta$

Recordamos lo visto en clase:

- Si $\boxed{\theta\ge\tfrac12}$, el método es **incondicionalmente estable** para la ecuación del calor: no aparece una restricción sobre $r=\Delta t/(\Delta x)^2$.
- Si $\boxed{0\le\theta<\tfrac12}$, la estabilidad requiere
$$\boxed{r\le\frac{1}{2(1-2\theta)}}.$$
- Para $\theta=0$ recuperamos Euler explícito y la condición clásica $r\le1/2$.
- Para $\theta=1$ recuperamos Euler implícito, que es incondicionalmente estable.
- Para $\theta=1/2$ obtenemos Crank--Nicolson, también incondicionalmente estable.

Ahora construiremos un mapa de calor. Para esto debemos construir, por ejemplo, una matriz que en cada columna (o fila) guarde los valores de $U$ a tiempo $t$ para luego visualizar la evolución temporal.

In [ ]:
function metodo_theta2(Uo,r,θ,N;steps = 1000)
    Δx = 1/N
    Δt = r*Δx^2
    J = round(Int, 1 / Δx) - 1
    x_grid = collect(1:J) .* Δx
    
    U = Uo(x_grid)
    
    A = Tridiagonal(ones(J-1), -2 .* ones(J), ones(J-1))
    B = I - θ * r * A
    C = I + (1 - θ) * r * A
    mapa = zeros(steps, J)

    for n in 1:steps            # FIX: antes era "n=1; while n<steps ... end", que dejaba la ultima fila (n=steps) sin llenar
        U = B \ (C * U)
        mapa[n, :] = U
    end

    p1 = plot(x_grid, U, xlabel="x", ylabel="u(x,t)", label="θ = $θ")
    p2 = heatmap(x_grid, (1:steps) .* Δt, mapa,
        xlabel="x", ylabel="t", colorbar_title="u")
    p = plot(p1, p2, layout=(1,2), size=(900,350))
    display(p)
    #println("El valor de r es: ", r)
    return U
end

U = metodo_theta2(Uo,0.3,1.0,50);

#### Orden de convergencia

Para medir el orden de convergencia espacial del método $\theta$ comparamos contra una solución exacta conocida. Tomando $u_0(x)=\sin(\pi x)$ (en vez de la parabólica), la solución exacta de
$$u_t=u_{xx},\quad u(0,t)=u(1,t)=0,\quad u(x,0)=\sin(\pi x)$$
es
$$u(x,t)=e^{-\pi^2 t}\sin(\pi x).$$

Refinamos $h$ manteniendo $r=\Delta t/h^2$ fijo (así $\Delta t$ se refina junto con $h$) y comparamos $U$ en el paso final contra la solución exacta evaluada en el mismo tiempo, usando la norma infinito. El orden se estima con la razón de logaritmos entre errores sucesivos, igual que hicimos con las fórmulas de diferenciación en la Clase 1.

In [ ]:
# Solución exacta de u_t = u_xx, u(0,t)=u(1,t)=0, u(x,0)=sin(πx)
sol_exacta(x, t) = exp(-π^2 * t) .* sin.(π .* x)

# Corre el método θ con u0=sin(πx) hasta el tiempo T y devuelve el error (norma ∞)
# contra la solución exacta, junto con el Δx usado.
function error_theta(θ, r, N; steps = 100)
    #completar
    Δx = 1 / N
    Δt = r * Δx^2
    pasos = round(Int, T / Δt)
    J = round(Int, 1 / Δx) - 1
    x_grid = collect(1:J) .* Δx

    U = sin.(π .* x_grid)
    A = Tridiagonal(ones(J-1), -2 .* ones(J), ones(J-1))
    B = I - θ * r * A
    C = I + (1 - θ) * r * A

    for n in 1:pasos
        U = B \ (C * U)
    end

    t_final = pasos * Δt
    error = norm(U - sol_exacta(x_grid, t_final), Inf)
    return error, Δx
end

# Refina N (y por lo tanto h y Δt, ya que r se mantiene fijo) y estima el orden
# de convergencia con la razón de logaritmos entre errores sucesivos.
function orden_convergencia(θ, r, T; Ns = [10, 20, 40, 80, 160])
    errores = Float64[]
    Δxs = Float64[]
    for N in Ns
        e, Δx = error_theta(θ, r, N, T)
        push!(errores, e)
        push!(Δxs, Δx)
    end

    ordenes = [log(errores[i]/errores[i+1]) / log(Δxs[i]/Δxs[i+1]) for i in 1:length(errores)-1]

    println("N\tΔx\t\terror\t\torden estimado")
    for i in eachindex(Ns)
        orden_str = i > 1 ? string(round(ordenes[i-1], digits=3)) : "-"
        println(Ns[i], "\t", round(Δxs[i], digits=5), "\t", round(errores[i], sigdigits=4), "\t", orden_str)
    end

    p = plot(Δxs, errores, xscale=:log10, yscale=:log10, marker=:circle,
        xlabel="Δx", ylabel="Error (norma ∞)", label="θ = $θ", legend=:topleft,
        title="Orden de convergencia (r = $r fijo)")
    referencia = Δxs.^2 .* (errores[1] / Δxs[1]^2)   # recta de referencia O(Δx²)
    plot!(p, Δxs, referencia, linestyle=:dash, label="O(Δx²)")
    display(p)

    return Δxs, errores, ordenes
end

orden_convergencia(0.5, 0.3, 0.05)

#### Para pensar:

Pensar ahora en el siguiente problema donde se agrega una fuente $f(x,t)$

$$\begin{align}
U_t&=U_{xx}+f(x,t)\\
U(0,t)&=U(1,t)=0\\
U(x,0)&=U_0(x)
\end{align}$$

¿Cómo lo implementarían?